# AgeXtend — Combined Pipeline for EvoKG


## Full Pipeline Overview

```
STEP 1 — Load shared reference files (PubChem, GO)
STEP 2 — Process 9 hallmark TSV files → Chemical–Hallmark CSVs       
STEP 3 — Process Agextend's drug-aging data → per-species CSVs          
STEP 4 — Standardize Chemical–Hallmark → Chemical–BiologicalProcess   
STEP 5 — Merge all hallmarks into combined Promotes/Inhibits/NoEffect  
STEP 6 — Process Hallmark–Hallmark relationships                       
```

## All Output Files

**From Step 2 (per-hallmark Chemical–Hallmark):**
CellularSenescence, AlteredIntercellularCommunication, DeregulatedNutrientSensing,
EpigeneticAlterations, GenomicInstability, LossOfProteostasis,
MitochondrialDysfunction, StemCellExhaustion, TelomereAttrition — all in `AgeXtend_Chemical_Hallmark/`

**From Step 3 (Sakshi's species-split CSVs):**
Human, Yeast, C.elegans (×2), Drosophila, Mouse — all in `Processed/`

**From Step 4–5 (standardized + merged):**
Per-hallmark Inhibits/Promotes/NoEffect CSVs + 3 combined files — all in `CHEM_BioP_Hallmarks/`

**From Step 6:**
`Aging_Hallmarks_BiologicalProcess_BiologicalProcess_Manullly.csv`


---
## STEP 0 — Path Configuration & Imports

In [3]:
import pandas as pd
import numpy as np
import os

In [2]:

# ============================================================
# PATH CONFIGURATION — update BASE_PATH before running
# ============================================================
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"

# Reference database paths
PUBCHEM_SYN_PATH  = f"{BASE_PATH}databases_for_mapping/pubchem/CID-Synonym-filtered"
PUBCHEM_PKL_PATH  = f"{BASE_PATH}databases_for_mapping/pubchem/combined_df.pkl"
GO_PATH           = f"{BASE_PATH}databases_for_mapping/MESH/MESH_GO_ID_NAME.csv"

# AgeXtend hallmark source TSV files (input for Step 2)
HALLMARK_TSV_DIR  = f"{BASE_PATH}agextend/AgeXtendHallmarksSourceFiles/HallmarksSourceFilesComplete"

# Sakshis collected drug-aging data (input for Step 3)
SAKSHI_DATA_PATH  = f"{BASE_PATH}/agextend/Sakshifinal_data.csv"

# Output directories
PROCESSED_DIR     = f"{OUT_PATH}agextend_processed"
HALLMARK_OUT_DIR  = f"{OUT_PATH}agextend_processed/AgeXtend_Chemical_Hallmark"   # Step 2 outputs / Step 4 inputs
BIOPRO_OUT_DIR    = f"{OUT_PATH}agextend_processed/CHEM_BioP_Hallmarks"          # Step 4–5 outputs
BIOPRO_OUT_DIR

# /storage/Arushi/090526_EvoAge/kg_formation/data_collection/Data_compile/AgeXtend/Processed'


# Create directories if they dont exist
os.makedirs(HALLMARK_OUT_DIR, exist_ok=True)
os.makedirs(BIOPRO_OUT_DIR,   exist_ok=True)

print("Directories ready.")
BIOPRO_OUT_DIR

# Create output directories if they donot exist
os.makedirs(HALLMARK_OUT_DIR, exist_ok=True)
os.makedirs(BIOPRO_OUT_DIR,   exist_ok=True)

print("Paths configured.")

Directories ready.
Paths configured.


---
## STEP 1 — Load Shared Reference Files
All three pipelines share the same PubChem dictionaries. Loading once here avoids redundant reloads.

In [3]:
# ── PubChem synonym → CID mapping ────────────────────────────────
# Used to resolve compound names to PubChem CIDs
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict       = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))  # synonym → CID
Pubchem_Syn_fil_dict_lower = {str(k).lower(): v for k, v in Pubchem_Syn_fil_dict.items()}  # lowercase version for case-insensitive lookup

# ── PubChem CID → IUPAC name and SMILES ──────────────────────────
Pubchem               = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))

# ── GO term ID → name mapping ─────────────────────────────────────
# Used in Step 4 to resolve GO IDs to human-readable hallmark names
GO = pd.read_csv(GO_PATH)
GO_dict              = dict(zip(GO['id'], GO['name']))
GO_id_namespace_dict = dict(zip(GO['id'], GO['namespace']))

# ── Aging hallmark GO ID mappings ────────────────────────────────
# Custom internal IDs (GO:111111x) were assigned where QuickGO had
# no official term. Cellular senescence (GO:0090398) is the only
# real QuickGO term in this set.
aging_hallmarks_to_go = {
    'genomic instability':                  'GO:1111111',
    'telomere attrition':                   'GO:1111112',
    'epigenetic alterations':               'GO:1111113',
    'loss of proteostasis':                 'GO:1111114',
    'deregulated nutrient sensing':         'GO:1111115',
    'mitochondrial dysfunction':            'GO:1111116',
    'cellular senescence':                  'GO:0090398',  # Official QuickGO term
    'stem cell exhaustion':                 'GO:1111117',
    'altered intercellular communication':  'GO:1111118'
}

# Reverse: GO ID → hallmark name (used in Step 4 for Tail_detail_name)
reversed_aging_hallmarks_dict = {v: k for k, v in aging_hallmarks_to_go.items()}

print(f"Loaded {len(Pubchem_Syn_fil_dict):,} PubChem synonym mappings")
print(f"Loaded {len(Pubchem_CID_Smile_Dict):,} SMILES mappings")
print(f"Loaded {len(GO_dict):,} GO term mappings")

Loaded 102,877,691 PubChem synonym mappings
Loaded 119,177,440 SMILES mappings
Loaded 47,995 GO term mappings


---
## STEP 2 — Process 9 Hallmark TSV Files → Chemical–Hallmark CSVs


Each TSV file contains compounds with a `Status` column:
- `1`  = Anti-effect (compound inhibits the hallmark)
- `0`  = No effect
- `-1` = Pro-effect (compound promotes the hallmark)

Outputs go to `AgeXtend_Chemical_Hallmark/` and are consumed by Step 4.

In [4]:
def process_hallmark_tsv(tsv_filename, tail_value, head_type, tail_type,
                         status_map, output_filename, extra_cols=None):
    """
    Process a single AgeXtend hallmark TSV → KG-ready Chemical–Hallmark CSV.

    Parameters
    ----------
    tsv_filename   : str   — source TSV filename in HALLMARK_TSV_DIR
    tail_value     : str   — hallmark name to assign as Tail
    head_type      : str   — entity type for Head ('ChemicalEntity' or 'Metabolite')
    tail_type      : str   — entity type for Tail ('Hallmark')
    status_map     : dict  — maps Status integers to relation label strings
    output_filename: str   — output CSV filename in HALLMARK_OUT_DIR
    extra_cols     : list  — additional columns to include (e.g. 'Name/Title/Synonyms')
    """
    # Load source TSV
    df = pd.read_csv(f"{HALLMARK_TSV_DIR}/{tsv_filename}", sep='\t')

    # Map Status integers to KG relation labels
    df['Status'] = df['Status'].replace(status_map)

    # Rename to KG schema
    df = df.rename(columns={'PUBCHEM_CID': 'Head', 'Status': 'Relation'})

    # Assign Tail and type metadata
    df['Tail']      = tail_value
    df['Head_type'] = head_type
    df['Tail_type'] = tail_type

    # Drop rows with missing Head or Tail
    df = df[~df['Head'].isna()]
    df = df[~df['Tail'].isna()]

    # Assign source and species (species not available at this level)
    df['Relation_Source'] = 'AgeXtend'
    df['Species']         = np.nan

    # Map PubChem CID → IUPAC name and SMILES
    df['Head']             = df['Head'].astype(str)
    df['Head_detail_name'] = df['Head'].map(Pubchem_IUPAC_CID_Dict)
    df['Head_SMILE_name']  = df['Head'].map(Pubchem_CID_Smile_Dict)

    # Set ID namespace labels
    df['Head_ID_IS'] = 'Pubchem'
    df['Tail_ID_IS'] = 'Hallmark'

    # Build column order; extra_cols inserted before 'Source'
    base_order = ['Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
                  'Head_detail_name', 'Head_SMILE_name', 'Relation_Source',
                  'Species', 'Head_ID_IS', 'Tail_ID_IS']
    extra      = extra_cols if extra_cols else []
    desired    = base_order + extra + (['Source'] if 'Source' in df.columns else [])
    existing   = [c for c in desired if c in df.columns]
    df         = df[existing].drop_duplicates()

    # Export
    df.to_csv(f"{HALLMARK_OUT_DIR}/{output_filename}", index=False)
    print(f"  {output_filename}: {len(df)} rows | Relations: {df['Relation'].value_counts().to_dict()}")
    return df

In [5]:
print("Processing hallmark TSV files...\n")

# Standard relation map used by all metabolite-type hallmarks
METABOLITE_STATUS_MAP = {
    1:  'Metabolite_AntiEffect_Hallmark',
    0:  'Metabolite_NoEffect',
    -1: 'Metabolite_ProEffect_Hallmark'
}

# 2.1 Cellular Senescence
# Uses ChemicalEntity (not Metabolite) and has an extra synonyms column
CS = process_hallmark_tsv(
    tsv_filename='celullar_senescence.tsv',
    tail_value='CellularSenescence',
    head_type='ChemicalEntity',
    tail_type='Hallmark',
    status_map={1: 'ChemicalEntity_AntiEffect_Hallmark',
                0: 'ChemicalEntity_NoEffect',
               -1: 'ChemicalEntity_ProEffect_Hallmark'},
    output_filename='CellularSenescence_AgeXtend.csv',
    extra_cols=['Name/Title/Synonyms']
)

# 2.2 Altered Intercellular Communication
AIC = process_hallmark_tsv(
    tsv_filename='altered_intercellular_comm.tsv',
    tail_value='AlteredIintercellularCommunication',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='AlteredIntercellularCommunication_AgeXtend.csv'
)

# 2.3 Deregulated Nutrient Sensing
DNS = process_hallmark_tsv(
    tsv_filename='dereg_nutrient_sensing.tsv',
    tail_value='DeregulatedNutrientSensing',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='DeregulatedNutrientSensing_AgeXtend.csv'
)

# 2.4 Epigenetic Alterations
EA = process_hallmark_tsv(
    tsv_filename='epigenetic_alterations.tsv',
    tail_value='EpigeneticAlterations',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='EpigeneticAlterations_AgeXtend.csv'
)

# 2.5 Genomic Instability
GI = process_hallmark_tsv(
    tsv_filename='genomic_instability.tsv',
    tail_value='GenomicInstability',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='GenomicInstability_AgeXtend.csv'
)

# 2.6 Loss of Proteostasis
LOP = process_hallmark_tsv(
    tsv_filename='loss_of_proteostasis.tsv',
    tail_value='LossOfProteostasis',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='LossOfProteostasis_AgeXtend.csv'
)

# 2.7 Mitochondrial Dysfunction
MD = process_hallmark_tsv(
    tsv_filename='mitochondrial_dysfunction.tsv',
    tail_value='MitochondrialDysfunction',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='MitochondrialDysfunction_AgeXtend.csv'
)

# 2.8 Stem Cell Exhaustion
SCE = process_hallmark_tsv(
    tsv_filename='stem_cell_exhaustion.tsv',
    tail_value='StemCellExhaustion',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='StemCellExhaustion_AgeXtend.csv'
)

# 2.9 Telomere Attrition
TA = process_hallmark_tsv(
    tsv_filename='telomere_attrition.tsv',
    tail_value='TelomereAttrition',
    head_type='Metabolite', tail_type='Hallmark',
    status_map=METABOLITE_STATUS_MAP,
    output_filename='TelomereAttrition_AgeXtend.csv'
)

print("\nStep 2 complete.")

Processing hallmark TSV files...

  CellularSenescence_AgeXtend.csv: 2549 rows | Relations: {'ChemicalEntity_AntiEffect_Hallmark': 1292, 'ChemicalEntity_NoEffect': 1174, 'ChemicalEntity_ProEffect_Hallmark': 83}
  AlteredIntercellularCommunication_AgeXtend.csv: 780 rows | Relations: {'Metabolite_AntiEffect_Hallmark': 452, 'Metabolite_ProEffect_Hallmark': 321, 'Metabolite_NoEffect': 7}
  DeregulatedNutrientSensing_AgeXtend.csv: 43299 rows | Relations: {'Metabolite_NoEffect': 42962, 'Metabolite_AntiEffect_Hallmark': 328, 'Metabolite_ProEffect_Hallmark': 9}
  EpigeneticAlterations_AgeXtend.csv: 41794 rows | Relations: {'Metabolite_ProEffect_Hallmark': 41372, 'Metabolite_AntiEffect_Hallmark': 422}
  GenomicInstability_AgeXtend.csv: 6077 rows | Relations: {'Metabolite_ProEffect_Hallmark': 3288, 'Metabolite_NoEffect': 2457, 'Metabolite_AntiEffect_Hallmark': 332}
  LossOfProteostasis_AgeXtend.csv: 199751 rows | Relations: {'Metabolite_NoEffect': 196581, 'Metabolite_AntiEffect_Hallmark': 2184, 

---
## STEP 3 — Process Sakshi's Drug-Aging Data → Per-Species CSVs


Sakshi's dataset contains manually collected drug-aging phenotype data from research papers.
Compound names are resolved to PubChem CIDs using a case-insensitive synonym lookup.
All rows are linked to GO:0007568 ('aging') as the Tail biological process.

In [6]:
# ── Load Sakshi's data ────────────────────────────────────────────
sakshi = pd.read_csv(SAKSHI_DATA_PATH)

# Rename columns to KG schema
sakshi.rename(columns={
    'Label':      'Tail',
    'Paper Link': 'Relation_Source',
    'smiles':     'Head_SEQ'
}, inplace=True)

# Map integer labels → aging phenotype categories
sakshi['Tail'] = sakshi['Tail'].replace({1: 'Anti-Aging', 0: 'Neutral-Aging'})

# Map phenotype categories → directional KG relation types
sakshi['Tail'] = sakshi['Tail'].replace({
    'ProAging':      'ChemicalEntity_Promotes_BiologicalProcess',
    'Anti-Aging':    'ChemicalEntity_Inhibits_BiologicalProcess',
    'Neutral-Aging': 'ChemicalEntity_NoEffect_BiologicalProcess'
})

print(f"Loaded {len(sakshi)} drug-aging entries")
print(f"Relation distribution:\n{sakshi['Tail'].value_counts()}")

Loaded 84 drug-aging entries
Relation distribution:
Tail
ChemicalEntity_Inhibits_BiologicalProcess    74
ChemicalEntity_NoEffect_BiologicalProcess    10
Name: count, dtype: int64


In [7]:
# ── Resolve compound names → PubChem CIDs ─────────────────────────
# Case-insensitive lookup; falls back to original name if no match
sakshi['Head'] = (
    sakshi['Compound Name'].str.lower()
    .map(Pubchem_Syn_fil_dict_lower)
    .fillna(sakshi['Compound Name'])
)
# Remove trailing '.0' from CID values (artifact of float conversion)
sakshi['Head'] = sakshi['Head'].astype(str).str.replace(r'\.0$', '', regex=True)

# Keep only target species
species_list = [
    'Drosophila melanogaster', 'Mus musculus',
    'Saccharomyces cerevisiae', 'Saccharomyces cerevisiae ',  # trailing space variant
    'Caenorhabditis elegans', 'Danio rerio',
    'Homo sapiens', 'Homo sapiens (cell culture)'
]
sakshi = sakshi[sakshi['Species'].isin(species_list)]

# Map PubChem CID → IUPAC name and SMILES
sakshi['Head_detail_name'] = sakshi['Head'].map(Pubchem_IUPAC_CID_Dict)
sakshi['Head_SMILE_name']  = sakshi['Head'].map(Pubchem_CID_Smile_Dict)

print(f"After species filter: {len(sakshi)} entries")
sakshi

After species filter: 79 entries


,Compound Name,IsomericSMILES,Relation_Source,Tail,Head_SEQ,Anti_Aging_Status,Anti_Aging_Prob,Species,Unnamed: 8,Head,Head_detail_name,Head_SMILE_name
0,Ligustilide,CCC/C=C\1/C2=C(C=CCC2)C(=O)O1,https://doi.org/10.1016/j.phymed.2023.155216,ChemicalEntity_Inhibits_BiologicalProcess,CCC/C=C/1\OC(=O)C2=C1CCC=C2,1,0.914950,Mus musculus,NaN,5319022,"(3Z)-3-butylidene-4,5-dihydro-2-benzofuran-1-one",CCC/C=C\1/C2=C(C=CCC2)C(=O)O1
1,Apigenin,C1=CC(=CC=C1C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O,https://doi.org/10.1016/j.mad.2023.111889,ChemicalEntity_Inhibits_BiologicalProcess,Oc1ccc(cc1)c1cc(=O)c2c(o1)cc(cc2O)O,1,0.920326,Mus musculus,NaN,5280443,"5,7-dihydroxy-2-(4-hydroxyphenyl)chromen-4-one",C1=CC(=CC=C1C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O
2,"2,5-anhydro-D-mannitol",C([C@@H]1[C@H]([C@@H]([C@H](O1)CO)O)O)O,https://doi.org/10.1186/s13062-024-00455-4,ChemicalEntity_Inhibits_BiologicalProcess,OC[C@H]1O[C@@H]([C@H]([C@@H]1O)O)CO,1,0.909696,Homo sapiens,NaN,73544,"(2R,3S,4S,5R)-2,5-bis(hydroxymethyl)oxolane-3,...",C([C@@H]1[C@H]([C@@H]([C@H](O1)CO)O)O)O
3,Plicamycin,C[C@@H]1[C@H]([C@@H](C[C@@H](O1)O[C@@H]2C[C@@H...,https://doi.org/10.1186/s13062-024-00455-4,ChemicalEntity_Inhibits_BiologicalProcess,CO[C@@H]([C@@H]1Cc2cc3cc(O[C@@H]4O[C@H](C)[C@H...,1,0.915105,Homo sapiens,NaN,163659,"(2S,3S)-2-[(2S,4R,5R,6R)-4-[(2S,4R,5S,6R)-4-[(...",C[C@@H]1[C@H]([C@@H](C[C@@H](O1)O[C@@H]2C[C@@H...
4,Nedaplatin,C(C(=O)O)O.N.N.[Pt],https://doi.org/10.1186/s13062-024-00455-4,ChemicalEntity_Inhibits_BiologicalProcess,OCC(=O)O.[Pt].N.N,1,0.856031,Homo sapiens,NaN,138402875,2-oxidoacetate;platinum(2+),C(C(=O)[O-])[O-].[Pt+2]
...,...,...,...,...,...,...,...,...,...,...,...,...
79,Hydroxylamine hydrochloride,NO.Cl,PMID: 24134630,ChemicalEntity_NoEffect_BiologicalProcess,NO.Cl,0,0.281992,Caenorhabditis elegans,NaN,443297,hydroxylamine;hydrochloride,NO.Cl
80,N-Bromoacetamide,CC(=O)NBr,PMID: 24134630,ChemicalEntity_NoEffect_BiologicalProcess,CC(=O)NBr,1,0.873322,Caenorhabditis elegans,NaN,4353,N-bromoacetamide,CC(=O)NBr
81,5-Fluorouracil,C1=C(C(=O)NC(=O)N1)F,PMID: 24134630,ChemicalEntity_NoEffect_BiologicalProcess,Fc1c[nH]c(=O)[nH]c1=O,0,0.238939,Caenorhabditis elegans,NaN,3385,"5-fluoro-1H-pyrimidine-2,4-dione",C1=C(C(=O)NC(=O)N1)F
82,Ebselen,C1=CC=C(C=C1)N2C(=O)C3=CC=CC=C3[Se]2,PMID: 24134630,ChemicalEntity_NoEffect_BiologicalProcess,O=c1c2ccccc2[se]n1c1ccccc1,1,0.908062,Caenorhabditis elegans,NaN,3194,"2-phenyl-1,2-benzoselenazol-3-one",C1=CC=C(C=C1)N2C(=O)C3=CC=CC=C3[Se]2


In [12]:
#


In [8]:
# ── Rename and assign KG metadata ─────────────────────────────────
sakshi = sakshi.rename(columns={
    'Tail':          'Relation',
    'Compound Name': 'Head_Alt_name',
    'Relation_Source': 'Data_Source'
})

# All rows link to GO:0007568 ('aging') as the BiologicalProcess Tail
sakshi['Tail_type']       = 'BiologicalProcess'
sakshi['Tail']            = 'GO:0007568'
sakshi['Tail_detail_name'] = 'aging'
sakshi['Relation_Source'] = 'AgeXtend'
sakshi['Head_type']       = 'ChemicalEntity'
sakshi['Data_type']       = 'Causative'

# Reorder columns; drop duplicates
desired_order = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Head_SMILE_name', 'Tail_detail_name',
    'Relation_Source', 'Species', 'Data_Source',
    'Anti_Aging_Status', 'Anti_Aging_Prob'
]
existing_cols = [c for c in desired_order if c in sakshi.columns]
sakshi = sakshi[existing_cols]
sakshi = sakshi.loc[:, ~sakshi.columns.duplicated()]

print(f"Final columns: {list(sakshi.columns)}")
print(f"Relation distribution:\n{sakshi['Relation'].value_counts()}")
sakshi

Final columns: ['Head', 'Relation', 'Tail', 'Head_type', 'Tail_type', 'Head_detail_name', 'Head_SMILE_name', 'Tail_detail_name', 'Relation_Source', 'Species', 'Data_Source', 'Anti_Aging_Status', 'Anti_Aging_Prob']
Relation distribution:
Relation
ChemicalEntity_Inhibits_BiologicalProcess    69
ChemicalEntity_NoEffect_BiologicalProcess    10
Name: count, dtype: int64


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Head_SMILE_name,Tail_detail_name,Relation_Source,Species,Data_Source,Anti_Aging_Status,Anti_Aging_Prob
0,5319022,ChemicalEntity_Inhibits_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"(3Z)-3-butylidene-4,5-dihydro-2-benzofuran-1-one",CCC/C=C\1/C2=C(C=CCC2)C(=O)O1,aging,AgeXtend,Mus musculus,https://doi.org/10.1016/j.phymed.2023.155216,1,0.914950
1,5280443,ChemicalEntity_Inhibits_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"5,7-dihydroxy-2-(4-hydroxyphenyl)chromen-4-one",C1=CC(=CC=C1C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O,aging,AgeXtend,Mus musculus,https://doi.org/10.1016/j.mad.2023.111889,1,0.920326
2,73544,ChemicalEntity_Inhibits_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"(2R,3S,4S,5R)-2,5-bis(hydroxymethyl)oxolane-3,...",C([C@@H]1[C@H]([C@@H]([C@H](O1)CO)O)O)O,aging,AgeXtend,Homo sapiens,https://doi.org/10.1186/s13062-024-00455-4,1,0.909696
3,163659,ChemicalEntity_Inhibits_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"(2S,3S)-2-[(2S,4R,5R,6R)-4-[(2S,4R,5S,6R)-4-[(...",C[C@@H]1[C@H]([C@@H](C[C@@H](O1)O[C@@H]2C[C@@H...,aging,AgeXtend,Homo sapiens,https://doi.org/10.1186/s13062-024-00455-4,1,0.915105
4,138402875,ChemicalEntity_Inhibits_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,2-oxidoacetate;platinum(2+),C(C(=O)[O-])[O-].[Pt+2],aging,AgeXtend,Homo sapiens,https://doi.org/10.1186/s13062-024-00455-4,1,0.856031
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,443297,ChemicalEntity_NoEffect_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,hydroxylamine;hydrochloride,NO.Cl,aging,AgeXtend,Caenorhabditis elegans,PMID: 24134630,0,0.281992
80,4353,ChemicalEntity_NoEffect_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,N-bromoacetamide,CC(=O)NBr,aging,AgeXtend,Caenorhabditis elegans,PMID: 24134630,1,0.873322
81,3385,ChemicalEntity_NoEffect_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"5-fluoro-1H-pyrimidine-2,4-dione",C1=C(C(=O)NC(=O)N1)F,aging,AgeXtend,Caenorhabditis elegans,PMID: 24134630,0,0.238939
82,3194,ChemicalEntity_NoEffect_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,"2-phenyl-1,2-benzoselenazol-3-one",C1=CC=C(C=C1)N2C(=O)C3=CC=CC=C3[Se]2,aging,AgeXtend,Caenorhabditis elegans,PMID: 24134630,1,0.908062


In [13]:
set((sakshi[sakshi['Species'] == 'Homo sapiens'])['Relation'])

{'ChemicalEntity_Inhibits_BiologicalProcess'}

In [14]:
set((sakshi[sakshi['Species'] == 'Saccharomyces cerevisiae'])['Relation'])

{'ChemicalEntity_Inhibits_BiologicalProcess'}

In [15]:
set((sakshi[sakshi['Species'] == 'Caenorhabditis elegans'])['Relation'])

{'ChemicalEntity_Inhibits_BiologicalProcess',
 'ChemicalEntity_NoEffect_BiologicalProcess'}

In [9]:
# ── Split by Species and Export ───────────────────────────────────

# Human
Human_df = sakshi[sakshi['Species'] == 'Homo sapiens']
Human_df.to_csv(f"{PROCESSED_DIR}/AgeXtend_Human_ChemicalEntity_Inhibits_BiologicalProcess.csv", index=False)
print(f"Human:      {len(Human_df)} rows")

# Yeast
Yeast_df = sakshi[sakshi['Species'] == 'Saccharomyces cerevisiae']
Yeast_df.to_csv(f"{PROCESSED_DIR}/AgeXtend_Yeast_ChemicalEntity_Inhibits_BiologicalProcess.csv", index=False)
print(f"Yeast:      {len(Yeast_df)} rows")

# C. elegans — split further by relation type
Cele_df      = sakshi[sakshi['Species'] == 'Caenorhabditis elegans']
Cele_Inhibits = Cele_df[Cele_df['Relation'] == 'ChemicalEntity_Inhibits_BiologicalProcess']
Cele_NoEffect = Cele_df[Cele_df['Relation'] == 'ChemicalEntity_NoEffect_BiologicalProcess']
Cele_Inhibits.to_csv(f"{PROCESSED_DIR}/AgeXtend_Cele_ChemicalEntity_Inhibits_BiologicalProcess.csv", index=False)
Cele_NoEffect.to_csv(f"{PROCESSED_DIR}/AgeXtend_Cele_ChemicalEntity_NoEffect_BiologicalProcess.csv", index=False)
print(f"C. elegans: Inhibits={len(Cele_Inhibits)}, NoEffect={len(Cele_NoEffect)}")

# Drosophila
Droso_df = sakshi[sakshi['Species'] == 'Drosophila melanogaster']
Droso_df.to_csv(f"{PROCESSED_DIR}/AgeXtend_Droso_ChemicalEntity_Inhibits_BiologicalProcess.csv", index=False)
print(f"Drosophila: {len(Droso_df)} rows")

# Mouse
Mouse_df = sakshi[sakshi['Species'] == 'Mus musculus']
Mouse_df.to_csv(f"{PROCESSED_DIR}/AgeXtend_Mouse_ChemicalEntity_Inhibits_BiologicalProcess.csv", index=False)
print(f"Mouse:      {len(Mouse_df)} rows")

print("\nStep 3 complete.")

Human:      15 rows
Yeast:      6 rows
C. elegans: Inhibits=30, NoEffect=10
Drosophila: 11 rows
Mouse:      7 rows

Step 3 complete.


---
## STEP 4 — Standardize Chemical–Hallmark → Chemical–BiologicalProcess
*(Originally Notebook 3)*

Reads the Step 2 CSVs, replaces raw relation labels with standardized
`ChemicalEntity_Inhibits/Promotes/NoEffect_BiologicalProcess` labels,
assigns proper GO IDs as Tail, and splits each hallmark into separate files.

In [22]:
def standardize_hallmark_to_biopro(input_filename, go_id, output_prefix,
                                   relation_map, filter_noeffect=False, extra_cols=None):
    """
    Standardize a Chemical–Hallmark CSV into Chemical–BiologicalProcess format.

    Parameters
    ----------
    input_filename  : str   — CSV filename in HALLMARK_OUT_DIR (Step 2 output)
    go_id           : str   — GO term ID to assign as Tail
    output_prefix   : str   — prefix for output filenames in BIOPRO_OUT_DIR
    relation_map    : dict  — maps Step 2 relation labels → standardized labels
    filter_noeffect : bool  — if True, drop NoEffect rows before processing (CS only)
    extra_cols      : list  — additional columns to carry through (e.g. 'Name/Title/Synonyms')
    """
    # Load Step 2 output
    df = pd.read_csv(f"{HALLMARK_OUT_DIR}/{input_filename}")

    # Optionally drop NoEffect rows (Cellular Senescence only)
    if filter_noeffect:
        df = df[df['Relation'] != 'ChemicalEntity_NoEffect']

    # Standardize relation labels to Inhibits/Promotes/NoEffect
    df['Relation'] = df['Relation'].replace(relation_map)

    # Assign GO ID as Tail and add BiologicalProcess metadata
    df['Head_type']        = 'ChemicalEntity'
    df['Tail']             = go_id
    df['Tail_detail_name'] = df['Tail'].map(reversed_aging_hallmarks_dict)
    df['Tail_type']        = 'BiologicalProcess'
    df['Tail_ID_IS']       = 'Quick_GO'

    # Reorder columns
    base = ['Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
            'Head_detail_name', 'Head_SMILE_name', 'Tail_detail_name',
            'Relation_Source', 'Species', 'Head_ID_IS', 'Tail_ID_IS']
    extra   = extra_cols if extra_cols else []
    desired = base + extra + (['Source'] if 'Source' in df.columns else [])
    existing = [c for c in desired if c in df.columns]
    df = df[existing]

    # Split by relation direction and export
    inhibits = df[df['Relation'] == 'ChemicalEntity_Inhibits_BiologicalProcess']
    promotes = df[df['Relation'] == 'ChemicalEntity_Promotes_BiologicalProcess']
    noeffect = df[df['Relation'] == 'ChemicalEntity_NoEffect_BiologicalProcess']

    inhibits.to_csv(f"{BIOPRO_OUT_DIR}/{output_prefix}_Chemical_Inhibits_BioPro.csv", index=False)
    promotes.to_csv(f"{BIOPRO_OUT_DIR}/{output_prefix}_Chemical_Promotes_BioPro.csv", index=False)
    if len(noeffect) > 0:
        noeffect.to_csv(f"{BIOPRO_OUT_DIR}/{output_prefix}_Chemical_NoEffect_BioPro.csv", index=False)

    print(f"  {output_prefix}: Inhibits={len(inhibits)}, Promotes={len(promotes)}, NoEffect={len(noeffect)}")
    return df

In [23]:
print("Standardizing hallmark CSVs to BiologicalProcess format...\n")

# Standard relation map for all metabolite-type hallmarks
METABOLITE_RELATION_MAP = {
    'Metabolite_AntiEffect_Hallmark': 'ChemicalEntity_Inhibits_BiologicalProcess',
    'Metabolite_ProEffect_Hallmark':  'ChemicalEntity_Promotes_BiologicalProcess',
    'Metabolite_NoEffect':            'ChemicalEntity_NoEffect_BiologicalProcess'
}

# 4.1 Cellular Senescence (GO:0090398 — only official QuickGO term)
# Filters NoEffect before processing; carries extra synonyms column
CS2 = standardize_hallmark_to_biopro(
    input_filename='CellularSenescence_AgeXtend.csv',
    go_id='GO:0090398',
    output_prefix='CellularSenescence', #CellularSensence',
    relation_map={'ChemicalEntity_AntiEffect_Hallmark': 'ChemicalEntity_Inhibits_BiologicalProcess',
                  'ChemicalEntity_ProEffect_Hallmark':  'ChemicalEntity_Promotes_BiologicalProcess'},
    filter_noeffect=True,
    extra_cols=['Name/Title/Synonyms']
)

# 4.2–4.9 Remaining 8 hallmarks (all use METABOLITE_RELATION_MAP)
AIC2 = standardize_hallmark_to_biopro(
    'AlteredIntercellularCommunication_AgeXtend.csv', 'GO:1111118',
    'AltInterCellComm', METABOLITE_RELATION_MAP)

DNS2 = standardize_hallmark_to_biopro(
    'DeregulatedNutrientSensing_AgeXtend.csv', 'GO:1111115',
    'DeregNutriSens', METABOLITE_RELATION_MAP)

EA2 = standardize_hallmark_to_biopro(
    'EpigeneticAlterations_AgeXtend.csv', 'GO:1111113',
    'EpigenAlter', METABOLITE_RELATION_MAP)

GI2 = standardize_hallmark_to_biopro(
    'GenomicInstability_AgeXtend.csv', 'GO:1111111',
    'GenomicInstability', METABOLITE_RELATION_MAP)

LOP2 = standardize_hallmark_to_biopro(
    'LossOfProteostasis_AgeXtend.csv', 'GO:1111114',
    'LossOfProteostasis', METABOLITE_RELATION_MAP)

MD2 = standardize_hallmark_to_biopro(
    'MitochondrialDysfunction_AgeXtend.csv', 'GO:1111116',
    'MitoDysfunc', METABOLITE_RELATION_MAP)

SCE2 = standardize_hallmark_to_biopro(
    'StemCellExhaustion_AgeXtend.csv', 'GO:1111117',
    'StemCellExhaust', METABOLITE_RELATION_MAP)

TA2 = standardize_hallmark_to_biopro(
    'TelomereAttrition_AgeXtend.csv', 'GO:1111112',
    'TelomereAttrition', METABOLITE_RELATION_MAP)

print("\nStep 4 complete.")

Standardizing hallmark CSVs to BiologicalProcess format...

  CellularSensence: Inhibits=1292, Promotes=83, NoEffect=0
  AltInterCellComm: Inhibits=452, Promotes=321, NoEffect=7
  DeregNutriSens: Inhibits=328, Promotes=9, NoEffect=42962
  EpigenAlter: Inhibits=422, Promotes=41372, NoEffect=0
  GenomicInstability: Inhibits=332, Promotes=3288, NoEffect=2457
  LossOfProteostasis: Inhibits=2184, Promotes=986, NoEffect=196581
  MitoDysfunc: Inhibits=871, Promotes=172, NoEffect=395263
  StemCellExhaust: Inhibits=669, Promotes=969, NoEffect=0
  TelomereAttrition: Inhibits=209, Promotes=136, NoEffect=5108

Step 4 complete.


---
## STEP 5 — Merge All Hallmarks into Combined Files
*(Originally Notebook 3)*

Concatenates all per-hallmark Inhibits/Promotes/NoEffect CSVs into
three combined files for downstream KG integration.

In [31]:
def merge_by_pattern(directory, pattern, output_name):
    """Concatenate all CSVs in directory matching a filename pattern and export."""
    files = [f for f in os.listdir(directory) if pattern in f and f.endswith('.csv')]
    if not files:
        print(f"  No files found for pattern '{pattern}'")
        return None
    dfs    = [pd.read_csv(os.path.join(directory, f)) for f in files]
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_csv(os.path.join(BIOPRO_OUT_DIR, output_name), index=False)
    print(f"  {output_name}: {len(merged)} rows from {len(files)} files")
    return merged

print("Merging all hallmark files...\n")

Chem_Promotes_BioPro = merge_by_pattern(BIOPRO_OUT_DIR, '_Promotes', 'All_Chemical_Promotes_BioPro(Hallmarks).csv')
Chem_Inhibits_BioPro = merge_by_pattern(BIOPRO_OUT_DIR, '_Inhibits', 'All_Chemical_Inhibits_BioPro(Hallmarks).csv')
Chem_NoEffect_BioPro = merge_by_pattern(BIOPRO_OUT_DIR, '_NoEffect_', 'All_Chemical_Noeffect_BioPro(Hallmarks).csv')

print("\nStep 5 complete.")

Merging all hallmark files...

  All_Chemical_Promotes_BioPro(Hallmarks).csv: 47336 rows from 9 files
  All_Chemical_Inhibits_BioPro(Hallmarks).csv: 6828 rows from 10 files
  All_Chemical_Noeffect_BioPro(Hallmarks).csv: 642378 rows from 6 files

Step 5 complete.


---
## STEP 6 — Hallmark–Hallmark (BiologicalProcess–BiologicalProcess) Relationships


Loads a manually curated TSV of hallmark-to-hallmark relationships,
maps GO IDs to names, and exports as CSV.

In [32]:
# Load manually curated hallmark–hallmark relationship file
BioPro_BioPro = pd.read_csv(
    f"{BASE_PATH}agextend/Aging_Hallmarks_BiologicalProcess_BiologicalProcess_Manullly.tsv", # Collected from literature Hallmarks of agibga 
    sep='\t'
)

# Map Tail GO IDs → human-readable hallmark names
BioPro_BioPro['Tail_Detail_name'] = BioPro_BioPro['Tail'].map(GO_dict)

# Export as CSV
BioPro_BioPro.to_csv(
    f"{PROCESSED_DIR}/Aging_Hallmarks_BiologicalProcess_BiologicalProcess_Manullly.csv",
    index=False
)

print(f"Hallmark–Hallmark relationships: {len(BioPro_BioPro)} rows")
print("\nStep 6 complete.")

Hallmark–Hallmark relationships: 94 rows

Step 6 complete.


In [9]:
BioPro_BioPro = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data/agextend_processed/Aging_Hallmarks_BiologicalProcess_BiologicalProcess_Manullly.csv')
BioPro_BioPro.head(5)

,Head,Relation,Tail,Head_type,Tail_type,Head_ID_IS,Tail_ID_IS,Head_Detail_name,Tail_Detail_name
0,GO:1111111,BiologicalProcess_BiologicalProcess,GO:0006281,BiologicalProcess,BiologicalProcess,Quick_GO,Quick_GO,genomic instability,DNA repair
1,GO:1111111,BiologicalProcess_BiologicalProcess,GO:0006302,BiologicalProcess,BiologicalProcess,Quick_GO,Quick_GO,genomic instability,double-strand break repair
2,GO:1111111,BiologicalProcess_BiologicalProcess,GO:0006298,BiologicalProcess,BiologicalProcess,Quick_GO,Quick_GO,genomic instability,mismatch repair
3,GO:1111111,BiologicalProcess_BiologicalProcess,GO:0006284,BiologicalProcess,BiologicalProcess,Quick_GO,Quick_GO,genomic instability,base-excision repair
4,GO:1111111,BiologicalProcess_BiologicalProcess,GO:0006289,BiologicalProcess,BiologicalProcess,Quick_GO,Quick_GO,genomic instability,nucleotide-excision repair
